In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import gc

# Only run the following code once and once you have it downloaded, then you don't need to run it again. 

In [ ]:
# Download latest version
# https://www.kaggle.com/datasets/ahmedshahriarsakib/usa-real-estate-dataset
path = kagglehub.dataset_download("ahmedshahriarsakib/usa-real-estate-dataset")

print("Path to dataset files:", path)

#### NOTE: You will need to either move the file downloaded in previous cell to another folder or configure your system to have read permission to where it is downloaded. <br> 
For this tutorial I just moved it. 

In [ ]:
df = pd.read_csv("../data/realtor-data.csv")

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.shape

I notice that there is some missing data by looking at the price, bed, bath, etc. <br>

So, there is a few things that we can learn from looking at this dataset and a few things that we can go about filling out this data.

In [ ]:
df = df.dropna(subset=['price'])

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
# ---------------------------
# Step 1: Load and preview data
# ---------------------------

# Example features and target
features = ['brokered_by', 'status', 'bed', 'bath', 'acre_lot', 'street', 'city', 'state', 'zip_code', 'house_size', 'prev_sold_date']
target = 'price'

X = df[features]
y = df[target]


In [ ]:

# ---------------------------
# Step 2: Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# ---------------------------
# Step 3: Create and train the model
# ---------------------------
model = LinearRegression()
model.fit(X_train, y_train)

Oh, this is where we need to know our models. <br> 

Linear regression wants to see numbers for the input. So, we need to adjust our original dataset by converting string columns to numbers and try again. 

In [ ]:
df_encoded = pd.get_dummies(df, columns=['city'], drop_first=True)


In [ ]:
df_encoded.head()

In [ ]:
df_encoded = pd.get_dummies(df_encoded, columns=['state'], drop_first=True)
df_encoded.head()

Noting that there is problems on the line above shows how columns can leave a dataset exploding into a REALLY large and sparse dataset. <br> 

So, let's start... again. 

In [ ]:
unique_statuses = df['status'].unique()
print(unique_statuses)

In [ ]:
#### Cleaning up memory with python's garbage collector. 
#del df_encoded
del X
del X_test
del X_train

del y
del y_test
del y_train


gc.collect()

Let's start with a simple model and then leave feature generation to a future iteration. We just need data. 

In [ ]:
# BACK TO Step 1: Load and preview data
# ---------------------------

# Example features and target
features = ['brokered_by', 'status', 'bed', 'bath', 'acre_lot', 'street', 'zip_code', 'house_size']
target = 'price'

# Quick mapping for first 
df['status'] = df['status'].map({
    'for_sale': 1,
    'ready_to_build': 2,
    'sold': 3
})

X = df[features]
y = df[target]

In [ ]:
# Validate that we only have ints/doubles for the dataset. No Strings. 
X

In [ ]:
# ---------------------------
# Not the best solution, for solving NaN's there are TONS of ways to handle this, but this is the simpliest. 
# ---------------------------

X = X.fillna(0)

In [ ]:
X

In [ ]:
# ---------------------------
# Step 2: Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# ---------------------------
# Step 3: Create and train the model
# ---------------------------
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:

# ---------------------------
# Step 4: Make predictions
# ---------------------------
y_pred = model.predict(X_test)

In [ ]:

# ---------------------------
# Step 5: Evaluate the model
# ---------------------------
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("R² Score:", r2)

In [ ]:
# ---------------------------
# Step 6: Plot predictions vs actual
# ---------------------------
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Home Prices")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()


In [ ]:
coeffs = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_
})
print(coeffs.sort_values(by='Coefficient', ascending=False))

# Next steps would be looking at variables and then working on improving the model.

# Ways that I would be looking at for improving the model.
1. Removing the outliars for the dataset to find a general representation. 
2. Improve data cleaning and figure out how I can add additional Features for the training dataset. 